# This is Jeopardy!

#### Overview

This project is slightly different than others you have encountered thus far. Instead of a step-by-step tutorial, this project contains a series of open-ended requirements which describe the project you'll be building. There are many possible ways to correctly fulfill all of these requirements, and you should expect to use the internet, Codecademy, and/or other resources when you encounter a problem that you cannot easily solve.

#### Project Goals

You will work to write several functions that investigate a dataset of _Jeopardy!_ questions and answers. Filter the dataset for topics that you're interested in, compute the average difficulty of those questions, and train to become the next Jeopardy champion!

## Prerequisites

In order to complete this project, you should have completed the Pandas lessons in the <a href="https://www.codecademy.com/learn/paths/analyze-data-with-python">Analyze Data with Python Skill Path</a>. You can also find those lessons in the <a href="https://www.codecademy.com/learn/data-processing-pandas">Data Analysis with Pandas course</a> or the <a href="https://www.codecademy.com/learn/paths/data-science/">Data Scientist Career Path</a>.

Finally, the <a href="https://www.codecademy.com/learn/practical-data-cleaning">Practical Data Cleaning</a> course may also be helpful.

## Project Requirements

1. We've provided a csv file containing data about the game show _Jeopardy!_ in a file named `jeopardy.csv`. Load the data into a DataFrame and investigate its contents. Try to print out specific columns.

   Note that in order to make this project as "real-world" as possible, we haven't modified the data at all - we're giving it to you exactly how we found it. As a result, this data isn't as "clean" as the datasets you normally find on Codecademy. More specifically, there's something odd about the column names. After you figure out the problem with the column names, you may want to rename them to make your life easier for the rest of the project.
   
   In order to display the full contents of a column, we've added this line of code for you:
   
   ```py
   pd.set_option('display.max_colwidth', None)
   ```

In [27]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
jeopardy = pd.read_csv('jeopardy.csv')
jeopardy.columns = jeopardy.columns.str.strip()
print(jeopardy.head(5))
print(jeopardy.columns)

   Show Number    Air Date      Round                         Category Value  \
0         4680  2004-12-31  Jeopardy!                          HISTORY  $200   
1         4680  2004-12-31  Jeopardy!  ESPN's TOP 10 ALL-TIME ATHLETES  $200   
2         4680  2004-12-31  Jeopardy!      EVERYBODY TALKS ABOUT IT...  $200   
3         4680  2004-12-31  Jeopardy!                 THE COMPANY LINE  $200   
4         4680  2004-12-31  Jeopardy!              EPITAPHS & TRIBUTES  $200   

                                                                                                      Question  \
0             For the last 8 years of his life, Galileo was under house arrest for espousing this man's theory   
1  No. 2: 1912 Olympian; football star at Carlisle Indian School; 6 MLB seasons with the Reds, Giants & Braves   
2                     The city of Yuma in this state has a record average of 4,055 hours of sunshine each year   
3                         In 1963, live on "The Art Linkletter 

2. Write a function that filters the dataset for questions that contains all of the words in a list of words. For example, when the list `["King", "England"]` was passed to our function, the function returned a DataFrame of 49 rows. Every row had the strings `"King"` and `"England"` somewhere in its `" Question"`.

   Test your function by printing out the column containing the question of each row of the dataset.

In [28]:
"""
def word_search(word_list, df):
    lowercase_word_list = []
    for word in word_list:
        lowercase_word_list.append(word.lower())
    lowercase_question = df['Question'].str.lower()
    #lowercase_answer = df['Answer'].str.lower()
    question_matches = lowercase_question.apply(lambda x: any(word in x for word in lowercase_word_list)) #this line checks if any of the words in the lowercase_word_list are present in each question, and returns a boolean Series where True indicates a match.
    #answer_matches = lowercase_answer.apply(lambda x: any(word in x for word in lowercase_word_list))
    return df[question_matches] #| answer_matches]
"""

def word_search_fast_exact(word_list, df):
    # 1. Add \b (word boundary) to both sides of each lowercase word
    # The 'r' before the string makes it a "raw string", which regex requires
    exact_words = [rf'\b{word.lower()}\b' for word in word_list]
    
    # 2. Join them with the OR operator (|)
    # If word_list is ['King', 'England'], this becomes: r'\bking\b|\bengland\b'
    search_pattern = '|'.join(exact_words) 
    
    # 3. Apply the mask (regex=True is default, but good to be explicit)
    mask = df['Question'].str.lower().str.contains(search_pattern, na=False, regex=True)
    
    return df[mask]

3. Test your original function with a few different sets of words to try to find some ways your function breaks. Edit your function so it is more robust.

   For example, think about capitalization. We probably want to find questions that contain the word `"King"` or `"king"`.
   
   You may also want to check to make sure you don't find rows that contain substrings of your given words. For example, our function found a question that didn't contain the word `"king"`, however it did contain the word `"viking"` &mdash; it found the `"king"` inside `"viking"`. Note that this also comes with some drawbacks &mdash; you would no longer find questions that contained words like `"England's"`.

In [29]:
king_england = word_search_fast_exact(['King', 'England'], jeopardy)
print(king_england.head(5))
print(len(king_england))
print(len(jeopardy))



     Show Number    Air Date             Round                    Category  \
40          4680  2004-12-31  Double Jeopardy!  DR. SEUSS AT THE MULTIPLEX   
50          4680  2004-12-31  Double Jeopardy!  DR. SEUSS AT THE MULTIPLEX   
116         5957  2010-07-06   Final Jeopardy!              HISTORIC WOMEN   
146         3751  2000-12-18  Double Jeopardy!           PEOPLE IN HISTORY   
296         4931  2006-02-06   Final Jeopardy!                FAMOUS SHIPS   

        Value  \
40      $1200   
50      $2000   
116  no value   
146      $200   
296  no value   

                                                                                                                                                                              Question  \
40   <a href="http://www.j-archive.com/media/2004-12-31_DJ_26.mp3">Ripped from today's headlines, he was a turtle king gone mad; Mack was the one good turtle who'd bring him down</a>   
50                 <a href="http://www.j-archive.com/med

4. We may want to eventually compute aggregate statistics, like `.mean()` on the `" Value"` column. But right now, the values in that column are strings. Convert the`" Value"` column to floats. If you'd like to, you can create a new column with float values.

   While most of the values in the `" Value"` column represent a dollar amount as a string, note that some do not &mdash; these values will need to be handled differently!

   Now that you can filter the dataset of question, use your new column that contains the float values of each question to find the "difficulty" of certain topics. For example, what is the average value of questions that contain the word `"King"`?
   
   Make sure to use the dataset that contains the float values as the dataset you use in your filtering function.

In [30]:
def string_to_float(dataframe, column_name):

# 1. Strip unwanted characters using regex
# This regex says: "Replace anything that is NOT (^) a digit (\d), period (.), or minus sign (-) with nothing ('')"
    cleaned_strings = dataframe[column_name].str.replace(r'[^\d.-]', '', regex=True)

# 2. Convert to float and handle the "no numeric values" issue
# errors='coerce' is the magic trick here. If it can't make a float, it returns NaN (Not a Number)
    dataframe['float_values'] = pd.to_numeric(cleaned_strings, errors='coerce')

string_to_float(jeopardy, 'Value')
print(jeopardy.head(5))

king_england_2 = word_search_fast_exact(['King', 'England'], jeopardy)
print(king_england_2.float_values.mean())


   Show Number    Air Date      Round                         Category Value  \
0         4680  2004-12-31  Jeopardy!                          HISTORY  $200   
1         4680  2004-12-31  Jeopardy!  ESPN's TOP 10 ALL-TIME ATHLETES  $200   
2         4680  2004-12-31  Jeopardy!      EVERYBODY TALKS ABOUT IT...  $200   
3         4680  2004-12-31  Jeopardy!                 THE COMPANY LINE  $200   
4         4680  2004-12-31  Jeopardy!              EPITAPHS & TRIBUTES  $200   

                                                                                                      Question  \
0             For the last 8 years of his life, Galileo was under house arrest for espousing this man's theory   
1  No. 2: 1912 Olympian; football star at Carlisle Indian School; 6 MLB seasons with the Reds, Giants & Braves   
2                     The city of Yuma in this state has a record average of 4,055 hours of sunshine each year   
3                         In 1963, live on "The Art Linkletter 

5. Write a function that returns the count of unique answers to all of the questions in a dataset. For example, after filtering the entire dataset to only questions containing the word `"King"`, we could then find all of the unique answers to those questions. The answer "Henry VIII" appeared 55 times and was the most common answer.

In [33]:
def unique_answers(dataframe, column_name):
    unique_values = dataframe[column_name].unique()
    return len(unique_values)

king_england_2_unique_answers = unique_answers(king_england_2, "Answer")
print(king_england_2_unique_answers)

2142


6. Explore from here! This is an incredibly rich dataset, and there are so many interesting things to discover. There are a few columns that we haven't even started looking at yet. Here are some ideas on ways to continue working with this data:

 * Investigate the ways in which questions change over time by filtering by the date. How many questions from the 90s use the word `"Computer"` compared to questions from the 2000s?
 * Is there a connection between the round and the category? Are you more likely to find certain categories, like `"Literature"` in Single Jeopardy or Double Jeopardy?
 * Build a system to quiz yourself. Grab random questions, and use the <a href="https://docs.python.org/3/library/functions.html#input">input</a> function to get a response from the user. Check to see if that response was right or wrong.

## Solution

7. Compare your program to our <a href="https://content.codecademy.com/PRO/independent-practice-projects/jeopardy/jeopardy_solution.zip">sample solution code</a> - remember, that your program might look different from ours (and probably will) and that's okay!

8. Great work! Visit <a href="https://discuss.codecademy.com/t/this-is-jeopardy-challenge-project-python-pandas/462365">our forums</a> to compare your project to our sample solution code. You can also learn how to host your own solution on GitHub so you can share it with other learners! Your solution might look different from ours, and that's okay! There are multiple ways to solve these projects, and you'll learn more by seeing others' code.